# 🕸️ Notebook 2 — A Tiny Async Event Bus

> **Goal:** Upgrade the toy bus from Notebook 1 into something that looks a *bit* more like a real broker: **asynchronous delivery** and **fault isolation** between subscribers.

We'll still fit it in 30 lines — the point is to feel the moving parts, not to compete with Kafka.


## 🛠️ Setup

```bash
cd 05-microservices/event-driven-architecture
uv sync
```

Then in VS Code, select the `.venv` kernel (top-right of the notebook).
If it isn't listed, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ A synchronous bus has two problems

The bus from Notebook 1 called handlers *inline*, in the producer's thread. That means:

1. **The producer waits** for every subscriber (we lose the "respond immediately" benefit).
2. **If a subscriber crashes, the next one never runs** and the exception bubbles up to the producer.

Let's prove it:


In [ ]:
from collections import defaultdict

class SyncBus:
    def __init__(self): self.subs = defaultdict(list)
    def subscribe(self, e, fn): self.subs[e].append(fn)
    def publish(self, e, p):
        for fn in self.subs[e]:
            fn(p)   # ❌ no isolation, no async

bus = SyncBus()

def flaky(p):   raise RuntimeError("💥 I always crash")
def shipping(p): print("  📦 shipping", p)

bus.subscribe("order_placed", flaky)
bus.subscribe("order_placed", shipping)

try:
    bus.publish("order_placed", {"order_id": 1})
except Exception as e:
    print("producer saw exception:", e)
print("did shipping run? ... no, it was never reached.")

## 2️⃣ The "best" version: queue + worker thread + per-handler try/except

Real brokers (Kafka, RabbitMQ, SNS/SQS) do three things our toy bus didn't:

1. **Buffer** events in a queue so the producer returns immediately.
2. **Deliver** events on a separate worker (or many workers).
3. **Isolate** subscriber failures so one bad handler doesn't poison the rest.

Here's the minimal version in pure Python — using `queue.Queue`, which is thread-safe (our old `deque` version had a subtle race).


In [ ]:
import queue, threading, time

class AsyncBus:
    def __init__(self):
        self.q = queue.Queue()
        self.subs = {}
        self._alive = True
        self._worker = threading.Thread(target=self._run, daemon=True)
        self._worker.start()

    def subscribe(self, event, handler):
        self.subs.setdefault(event, []).append(handler)

    def publish(self, event, payload):
        self.q.put((event, payload))   # ✅ returns instantly

    def _run(self):
        while self._alive:
            try:
                event, payload = self.q.get(timeout=0.1)
            except queue.Empty:
                continue
            for fn in list(self.subs.get(event, [])):
                try:
                    fn(payload)
                except Exception as e:
                    # ✅ isolation: one crash doesn't stop the others
                    print(f"  ⚠️  subscriber {fn.__name__} crashed: {e}")

    def stop(self):
        self._alive = False

bus = AsyncBus()

## 3️⃣ Start an e-commerce workflow

We'll wire up two subscribers and publish an order. Note the producer returns on the same line — the handlers run *after*.


In [ ]:
def shipping(p): print(f"  📦 ship    order={p['order_id']}")
def email(p):    print(f"  ✉️  email   {p['email']}")

bus.subscribe("order_placed", shipping)
bus.subscribe("order_placed", email)

print("producer: publishing...")
bus.publish("order_placed", {"order_id": 1, "email": "ada@example.com", "total": 42})
print("producer: done (not blocked!)")
time.sleep(0.2)   # give the worker a moment to drain the queue

## 4️⃣ Add new features *without touching the producer*

This is the money-shot of event-driven architecture. Marketing wants loyalty points.
Security wants a fraud alert for big orders. Neither team needs to change `place_order`.


In [ ]:
def fraud(p):
    if p["total"] > 1000:
        print(f"  🚨 fraud review for order={p['order_id']} total=${p['total']}")

def loyalty(p):
    pts = int(p["total"])
    print(f"  ⭐ +{pts} pts -> {p['email']}")

bus.subscribe("order_placed", fraud)
bus.subscribe("order_placed", loyalty)

bus.publish("order_placed", {"order_id": 2, "email": "grace@example.com", "total": 1500})
time.sleep(0.2)

## 5️⃣ Prove fault isolation

Now we'll add a deliberately broken subscriber. Unlike section 1, the others keep running.


In [ ]:
def broken(p):
    raise RuntimeError("KABOOM")

bus.subscribe("order_placed", broken)

bus.publish("order_placed", {"order_id": 3, "email": "alan@example.com", "total": 250})
time.sleep(0.2)
print("producer is still alive, shipping/email/loyalty all ran.")

## 🧠 Benefits vs costs — cheatsheet

**Benefits you just felt**
- 🧩 New features = new subscribers (producer untouched).
- 🧯 A bad subscriber doesn't take the system down.
- ⚡ Producer latency is *publish time*, not *sum of consumers*.
- 🎚️ Consumers can be scaled independently (imagine running `loyalty` on 10 workers).

**Costs you now need to manage**
- 🕵️ Flow is implicit — you need good logging/tracing to debug (add an `event_id`!).
- 📜 The event name + payload shape is a **public contract**. Change it carelessly and silent consumers break.
- 🔁 Real brokers deliver **at-least-once** — the same event can arrive twice. Handlers must be **idempotent**.
- 🧮 Ordering across partitions/topics is rarely guaranteed.

👉 Notebook 3 shows **how to design events** so these costs stay manageable: facts vs commands, schema evolution, idempotent consumers, and broker-vs-mediator topologies.
